# Gradient Boosting Regressor

Dự đoán trực tiếp `SalePrice`. Preprocessing nằm trong Pipeline để tránh data leakage.

In [6]:
import sys
from pathlib import Path
import json
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_preprocessing import load_house_prices, prepare_features, build_preprocessor
from src.evaluation import regression_metrics, save_metrics
from src.experiment_tracking import start_experiment, save_model, log_wandb

In [7]:
DATA_DIR = PROJECT_ROOT / 'data'
EXPERIMENT_ROOT = PROJECT_ROOT / 'experiments'
RANDOM_STATE = 42
train, test = load_house_prices(DATA_DIR)
X, y, X_test = prepare_features(train, test)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
preprocessor, numeric_features, categorical_features = build_preprocessor(X_train)
print(X_train.shape, X_valid.shape, X_test.shape)

(1168, 75) (292, 75) (1459, 75)


In [8]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(random_state=RANDOM_STATE)),
])

param_grid = {
    "regressor__n_estimators": [100, 300],
    "regressor__learning_rate": [0.03, 0.05, 0.1],
    "regressor__max_depth": [2, 3],
    "regressor__min_samples_leaf": [2, 5],
}

search = GridSearchCV(
    pipeline,
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    refit=True,
)
search.fit(X_train, y_train)
valid_pred = search.predict(X_valid)
metrics = regression_metrics(y_valid, valid_pred)
print("Best parameters:", search.best_params_)
print(json.dumps(metrics, indent=2))

Best parameters: {'regressor__learning_rate': 0.1, 'regressor__max_depth': 3, 'regressor__min_samples_leaf': 2, 'regressor__n_estimators': 300}
{
  "mae": 15932.272119361221,
  "rmse": 26367.22115792624,
  "r2": 0.9093610450440087,
  "rmsle": 0.1364356048623695
}


In [9]:
import os
os.environ['WANDB_MODE'] = 'online'

run_id, run_dir = start_experiment(
    EXPERIMENT_ROOT,
    "gradient_boosting",
    {
        "model": "GradientBoostingRegressor",
        "random_state": RANDOM_STATE,
        "cv": 5,
        "best_params": search.best_params_,
    },
)
final_model = search.best_estimator_.fit(X, y)
test_pred = final_model.predict(X_test)
prediction_path = run_dir / "predictions.csv"
pd.DataFrame({"Id": test["Id"], "SalePrice": test_pred}).to_csv(prediction_path, index=False)
metrics_path = run_dir / "metrics.json"
save_metrics(metrics, metrics_path)
model_path = run_dir / "model.pkl"
save_model(final_model, model_path)
mode = log_wandb(
    run_dir,
    "house-price-regression",
    "gradient_boosting",
    {"best_params": search.best_params_},
    metrics,
    [model_path, prediction_path, metrics_path, run_dir / "config.json"],
)
print(f"Run: {run_dir}")
print(f"W&B mode: {mode}")

mae,▁
r2,▁
rmse,▁
rmsle,▁
mae,15932.27212
r2,0.90936
rmse,26367.22116
rmsle,0.13644


Run: c:\Users\ASUS\Data_Science_ Junior\DL\Tuan02\house_price\experiments\gradient_boosting\20260919_180037
W&B mode: online
